# Recession Impact Analysis
## Predictive Modeling & Future Hypothesis

Tests whether recession indicators can forecast sector job losses.

**Hook #5:** If a recession hits tomorrow, which cohort faces a steeper percentage wipeout—and how many months to recover?

---


### Step 1: Load tools and data

Reads `master_table.csv` from Notebook 1.


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

plt.rcParams["figure.figsize"] = (12, 5)

df = pd.read_csv(os.path.join("..", "..", "data", "processed", "master_table.csv"),
                 parse_dates=["date"], index_col="date").sort_index()

print(f"Loaded {df.shape[0]} rows x {df.shape[1]} columns")
print(f"Date range: {df.index.min().date()} to {df.index.max().date()}")

### Step 2: Feature engineering

Targets: forward 12-month % change for blue_collar_jobs and white_collar_jobs.
Features: recession indicator, lagged employment levels, lagged ratios.
Time-based split: first 80% train, last 20% test.


In [ ]:
emp_cols = ["blue_collar_jobs", "white_collar_jobs"]
model_df = df.copy()

# Forward 12-month % change targets
for col in emp_cols:
    model_df[f"{col}_fwd12_pct"] = model_df[col].shift(-12) / model_df[col] - 1

# Lagged features
for col in emp_cols:
    for lag in [1, 3, 6, 12]:
        model_df[f"{col}_lag{lag}"] = model_df[col].shift(lag)

model_df["recession_indicator"] = df["recession_indicator"]
model_df["white_collar_pct_lag1"] = df["white_collar_pct"].shift(1)
model_df["blue_collar_pct_lag1"] = df["blue_collar_pct"].shift(1)

model_df = model_df.dropna()
print(f"Model-ready: {model_df.shape[0]} rows x {model_df.shape[1]} columns")

### Step 3: Train models

One linear regression per cohort. Time-respecting split (no shuffling).


In [ ]:
feature_cols = ["recession_indicator", "white_collar_pct_lag1", "blue_collar_pct_lag1"] + [
    f"{c}_lag{l}" for c in emp_cols for l in [1, 3, 6, 12]
]
results = []
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, col in enumerate(emp_cols):
    target = f"{col}_fwd12_pct"
    X = model_df[feature_cols]
    y = model_df[target]
    split = int(len(X) * 0.8)
    
    m = LinearRegression()
    m.fit(X.iloc[:split], y.iloc[:split])
    pred = m.predict(X.iloc[split:])
    
    rmse = np.sqrt(mean_squared_error(y.iloc[split:], pred))
    r2 = r2_score(y.iloc[split:], pred)
    rec_coef = m.coef_[feature_cols.index("recession_indicator")]
    
    results.append({
        "cohort": col,
        "rmse": round(rmse, 4),
        "r2": round(r2, 4),
        "recession_coef": round(rec_coef, 4),
    })
    
    axes[i].scatter(y.iloc[split:], pred, alpha=0.6, s=20)
    axes[i].plot([y.min(), y.max()], [y.min(), y.max()], "r--", lw=1)
    axes[i].set_xlabel("Actual Forward 12M YoY")
    axes[i].set_ylabel("Predicted Forward 12M YoY")
    axes[i].set_title(f"{col.replace('_', ' ').title()}\nR² = {r2:.3f}", fontweight="bold")
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

display(pd.DataFrame(results).set_index("cohort"))

### Step 5: Recovery duration analysis

Historical table: how many months until employment returns to pre-recession peak,
by cohort and by recession.


In [ ]:
recession_starts = df[df["recession_indicator"] == 1].index
recession_starts = recession_starts[recession_starts.to_series().diff().dt.days > 60]

recovery_results = []
for t0 in recession_starts:
    base_blue = df.loc[t0, "blue_collar_jobs"]
    base_white = df.loc[t0, "white_collar_jobs"]
    
    for cohort, base_val in [("blue_collar", base_blue), ("white_collar", base_white)]:
        col = f"{cohort}_jobs"
        rec_end = df.loc[df.index >= t0][df["recession_indicator"] == 0].index.min()
        if pd.isna(rec_end):
            rec_end = df.index[-1]
        rec_period = df.loc[t0:rec_end][col]
        if len(rec_period) == 0:
            continue
        
        # Find first month after recession end where employment >= pre-recession peak
        post_rec = df.loc[df.index > t0][col]
        recovery_months = None
        for date in post_rec.index:
            if post_rec[date] >= base_val:
                recovery_months = (date.year - t0.year) * 12 + (date.month - t0.month)
                break
        
        max_drop = ((rec_period.min() / base_val - 1) * 100)
        recovery_results.append({
            "Recession": t0.date(),
            "Cohort": cohort,
            "Max Drop %": round(max_drop, 1),
            "Recovery Months": recovery_months,
        })

recovery_df = pd.DataFrame(recovery_results)
print(recovery_df.pivot(index="Cohort", columns="Recession", values=["Max Drop %", "Recovery Months"]))
print("\nAverage recovery months by cohort:")
print(recovery_df.groupby("Cohort")["Recovery Months"].mean())

### Step 6: Hook #5 — Answers

Which cohort faces a steeper wipeout, and how long to recover?


In [ ]:
results_df = pd.DataFrame(results)
print("=== MODEL RESULTS: Forward 12-Month % Change ===")
print(results_df.set_index("cohort"))
print()
print("=== RECOVERY ANALYSIS ===")
print(recovery_df.groupby("Cohort")["Recovery Months"].agg(["mean", "min", "max"]))
print()

# Determine which cohort is more vulnerable
blue_coef = results_df[results_df["cohort"] == "blue_collar_jobs"]["recession_coef"].iloc[0]
white_coef = results_df[results_df["cohort"] == "white_collar_jobs"]["recession_coef"].iloc[0]

print("=== HOOK #5 ANSWERS ===")
if blue_coef < white_coef:
    print("1. Steeper wipeout: BLUE-COLLAR jobs")
    print(f"   Recession coefficient: Blue={blue_coef:.4f}, White={white_coef:.4f}")
else:
    print("1. Steeper wipeout: WHITE-COLLAR jobs")
    print(f"   Recession coefficient: White={white_coef:.4f}, Blue={blue_coef:.4f}")

avg_rec_blue = recovery_df[recovery_df["Cohort"] == "blue_collar"]["Recovery Months"].mean()
avg_rec_white = recovery_df[recovery_df["Cohort"] == "white_collar"]["Recovery Months"].mean()
if pd.notna(avg_rec_blue) and pd.notna(avg_rec_white):
    if avg_rec_blue > avg_rec_white:
        print("2. Slower recovery: BLUE-COLLAR jobs")
        print(f"   Avg recovery: Blue={avg_rec_blue:.1f} months, White={avg_rec_white:.1f} months")
    else:
        print("2. Slower recovery: WHITE-COLLAR jobs")
        print(f"   Avg recovery: White={avg_rec_white:.1f} months, Blue={avg_rec_blue:.1f} months")
print()
print("3. Model R² values show how much of future job loss the recession indicator + lags explain.")
print("   Higher R² = more predictable. Lower R² = more idiosyncratic shocks.")